In [ ]:
# imports

import pandas as pd
import plotly.express as px

In [ ]:
from pathlib import Path

# Folder holding one exported CSV per listener
DATA_DIR = Path("data")

# Encodings to try in order; utf-8-sig also strips the BOM some exports have
ENCODINGS = ["utf-8-sig", "utf-16", "cp1252", "latin-1"]


def read_csv_any_encoding(path):
    """Read a CSV, trying each encoding until one works."""
    for encoding in ENCODINGS:
        try:
            return pd.read_csv(path, encoding=encoding)
        except (UnicodeError, pd.errors.ParserError):
            # Wrong encoding for this file, try the next one
            continue
    raise ValueError(f"Could not read {path} with any of {ENCODINGS}")


# Load every CSV in the data folder and tag its rows with the listener
frames = []
for path in sorted(DATA_DIR.glob("*.csv")):
    df = read_csv_any_encoding(path)
    # Listener name is the first word of the file name,
    # e.g. "Logan Exported Data.csv" -> "Logan"
    df["Listener"] = path.stem.split()[0]
    frames.append(df)

# Stack all listeners into one DataFrame
data = pd.concat(frames, ignore_index=True)

# Quick sanity check: total size and rows per listener
print(data.shape)
print(data["Listener"].value_counts())
data.head()

(3197, 25)
Listener
Amadou    1663
Logan     1534
Name: count, dtype: int64


,Track URI,Track Name,Album Name,Artist Name(s),Release Date,Duration (ms),Popularity,Explicit,Added By,Added At,...,Loudness,Mode,Speechiness,Acousticness,Instrumentalness,Liveness,Valence,Tempo,Time Signature,Listener
0,spotify:track:6kWxIqQDsKFYCJGbU4AjCX,Hardest To Love,After Hours,The Weeknd,2020-03-20,211400,68,True,tcgbb1ix1h1h974y7pakbtpr2,2026-09-01T20:30:03Z,...,-7.847,0.0,0.0713,0.0909,0.000088,0.125,0.111,81.898,4.0,Amadou
1,spotify:track:5Y77SQxEr1eiofPeUTPHxM,You,Street Love,Lloyd;Lil Wayne,2007-02-12,273133,86,False,tcgbb1ix1h1h974y7pakbtpr2,2026-09-01T20:30:03Z,...,-6.008,0.0,0.4720,0.1020,0.000000,0.297,0.706,130.210,5.0,Amadou
2,spotify:track:1mhVXWduD8ReDwusfaHNwU,Tell Your Friends,Beauty Behind The Madness,The Weeknd,2015-08-28,334333,71,True,tcgbb1ix1h1h974y7pakbtpr2,2026-09-01T20:30:03Z,...,-11.091,0.0,0.0425,0.2650,0.000000,0.181,0.309,74.640,4.0,Amadou
3,spotify:track:058gdiBln6OADguG08GOaU,Tru,TRU - LP,Lloyd,2018-08-31,255640,64,True,tcgbb1ix1h1h974y7pakbtpr2,2026-09-01T20:30:03Z,...,-5.877,1.0,0.0670,0.1190,0.000000,0.117,0.135,143.358,4.0,Amadou
4,spotify:track:6n3HGiq4v35D6eFOSwqYuo,Teenage Fever,More Life,Drake,2017-03-18,219692,84,True,tcgbb1ix1h1h974y7pakbtpr2,2026-09-01T20:30:03Z,...,-10.330,0.0,0.0846,0.1110,0.017200,0.115,0.144,97.040,4.0,Amadou


In [ ]:
data.groupby("Listener").size()

Listener
Amadou    1663
Logan     1534
dtype: int64

In [ ]:
data.columns.tolist()

['Track URI',
 'Track Name',
 'Album Name',
 'Artist Name(s)',
 'Release Date',
 'Duration (ms)',
 'Popularity',
 'Explicit',
 'Added By',
 'Added At',
 'Genres',
 'Record Label',
 'Danceability',
 'Energy',
 'Key',
 'Loudness',
 'Mode',
 'Speechiness',
 'Acousticness',
 'Instrumentalness',
 'Liveness',
 'Valence',
 'Tempo',
 'Time Signature',
 'Listener']

In [ ]:
data.shape

(3197, 25)

In [ ]:
data.groupby("Listener")["Genres"].apply(lambda s: s.isna().sum())

Listener
Amadou    569
Logan     421
Name: Genres, dtype: int64

In [ ]:
(
    data.groupby("Listener")["Genres"]
    .value_counts(dropna=False)
    .groupby(level=0)
    .head(20)
)

Listener  Genres                                                              
Amadou    NaN                                                                     569
          r&b                                                                     106
          rap                                                                     105
          melodic rap                                                              68
          melodic rap,rage rap                                                     39
          bedroom pop                                                              33
          melodic rap,rap                                                          25
          cloud rap                                                                25
          rage rap                                                                 19
          alternative r&b                                                          17
          alternative r&b,neo soul                           

In [ ]:
features = [
    "Valence",
    "Energy",
    "Danceability",
    "Acousticness",
    "Tempo",
    "Loudness",
    "Popularity"
]

data.groupby("Listener")[features].describe()

Valence                                                              \
           count      mean       std      min      25%     50%    75%    max   
Listener                                                                       
Amadou    1662.0  0.486498  0.235306  0.00001  0.30200  0.4820  0.672  0.980   
Logan     1534.0  0.453013  0.215721  0.00000  0.28925  0.4245  0.600  0.969   

          Energy            ... Loudness        Popularity             \
           count      mean  ...      75%    max      count       mean   
Listener                    ...                                         
Amadou    1662.0  0.600632  ...  -5.6505  0.832     1663.0  57.529164   
Logan     1534.0  0.706849  ...  -4.4155 -1.190     1534.0  53.427640   

                                                  
                std  min   25%   50%   75%   max  
Listener                                          
Amadou    25.859897  0.0  50.0  65.0  75.0  98.0  
Logan     25.171343  0.0  40.0  58.0  73.0  96.0  

[2 rows x 56 columns]

In [ ]:
data.head()

,Track URI,Track Name,Album Name,Artist Name(s),Release Date,Duration (ms),Popularity,Explicit,Added By,Added At,...,Loudness,Mode,Speechiness,Acousticness,Instrumentalness,Liveness,Valence,Tempo,Time Signature,Listener
0,spotify:track:6kWxIqQDsKFYCJGbU4AjCX,Hardest To Love,After Hours,The Weeknd,2020-03-20,211400,68,True,tcgbb1ix1h1h974y7pakbtpr2,2026-09-01T20:30:03Z,...,-7.847,0.0,0.0713,0.0909,0.000088,0.125,0.111,81.898,4.0,Amadou
1,spotify:track:5Y77SQxEr1eiofPeUTPHxM,You,Street Love,Lloyd;Lil Wayne,2007-02-12,273133,86,False,tcgbb1ix1h1h974y7pakbtpr2,2026-09-01T20:30:03Z,...,-6.008,0.0,0.4720,0.1020,0.000000,0.297,0.706,130.210,5.0,Amadou
2,spotify:track:1mhVXWduD8ReDwusfaHNwU,Tell Your Friends,Beauty Behind The Madness,The Weeknd,2015-08-28,334333,71,True,tcgbb1ix1h1h974y7pakbtpr2,2026-09-01T20:30:03Z,...,-11.091,0.0,0.0425,0.2650,0.000000,0.181,0.309,74.640,4.0,Amadou
3,spotify:track:058gdiBln6OADguG08GOaU,Tru,TRU - LP,Lloyd,2018-08-31,255640,64,True,tcgbb1ix1h1h974y7pakbtpr2,2026-09-01T20:30:03Z,...,-5.877,1.0,0.0670,0.1190,0.000000,0.117,0.135,143.358,4.0,Amadou
4,spotify:track:6n3HGiq4v35D6eFOSwqYuo,Teenage Fever,More Life,Drake,2017-03-18,219692,84,True,tcgbb1ix1h1h974y7pakbtpr2,2026-09-01T20:30:03Z,...,-10.330,0.0,0.0846,0.1110,0.017200,0.115,0.144,97.040,4.0,Amadou


In [ ]:
fig = px.scatter(
    data,
    x="Valence",
    y="Energy",
    facet_col="Listener",
    hover_data=["Track Name", "Artist Name(s)", "Genres"],
    title="Valence vs Energy of Songs"
)

fig.show()

The graph shows that songs with higher valence often also have higher energy, but there is still a lot of variation. This suggests that valence and energy are somewhat related, but they are measuring different characteristics of a song.

In [ ]:
data.groupby("Listener")[features].corr()

Valence    Energy  Danceability  Acousticness  \
Listener                                                                
Amadou   Valence       1.000000  0.364301      0.310308     -0.061158   
         Energy        0.364301  1.000000      0.003557     -0.487094   
         Danceability  0.310308  0.003557      1.000000     -0.159155   
         Acousticness -0.061158 -0.487094     -0.159155      1.000000   
         Tempo         0.031895  0.096156     -0.055978     -0.116550   
         Loudness      0.230355  0.713101      0.127227     -0.448445   
         Popularity    0.042512  0.040259     -0.091688     -0.016125   
Logan    Valence       1.000000  0.277427      0.486616     -0.194453   
         Energy        0.277427  1.000000     -0.060970     -0.802835   
         Danceability  0.486616 -0.060970      1.000000      0.012872   
         Acousticness -0.194453 -0.802835      0.012872      1.000000   
         Tempo         0.099903  0.231442     -0.172678     -0.201750   
         Loudness      0.135712  0.737254     -0.022663     -0.602215   
         Popularity    0.125026  0.045998      0.171779     -0.082104   

                          Tempo  Loudness  Popularity  
Listener                                               
Amadou   Valence       0.031895  0.230355    0.042512  
         Energy        0.096156  0.713101    0.040259  
         Danceability -0.055978  0.127227   -0.091688  
         Acousticness -0.116550 -0.448445   -0.016125  
         Tempo         1.000000  0.059827   -0.063337  
         Loudness      0.059827  1.000000    0.070393  
         Popularity   -0.063337  0.070393    1.000000  
Logan    Valence       0.099903  0.135712    0.125026  
         Energy        0.231442  0.737254    0.045998  
         Danceability -0.172678 -0.022663    0.171779  
         Acousticness -0.201750 -0.602215   -0.082104  
         Tempo         1.000000  0.187201   -0.019819  
         Loudness      0.187201  1.000000    0.001631  
         Popularity   -0.019819  0.001631    1.000000

The results show that energy and loudness have the strongest relationship, so songs with more energy also tend to be louder. Energy also decreases as acousticness increases. Valence and energy have a positive but moderate relationship, which supports what we saw in the scatterplot. Popularity does not seem to have a strong relationship with the other audio features.

In [ ]:
fig = px.scatter(
    data,
    x="Energy",
    y="Loudness",
    facet_col="Listener",
    hover_data=["Track Name", "Artist Name(s)", "Genres"],
    title="Energy vs Loudness of Songs"
)

fig.show()

I made this graph because the correlation results showed that energy and loudness had the strongest relationship in our dataset. The graph makes that relationship easier to see: as songs become more energetic, they generally also become louder. This gives us another useful pair of features to consider when comparing songs.

In [ ]:
genre_df = data.dropna(subset=["Genres"]).copy()

genre_df["Genre"] = genre_df["Genres"].str.split(",")
genre_df = genre_df.explode("Genre")
genre_df["Genre"] = genre_df["Genre"].str.strip()

# Top 5 genres for each listener separately
top_genres = (
    genre_df.groupby("Listener")["Genre"]
    .value_counts()
    .groupby(level=0)
    .head(5)
)
top_genres

Listener  Genre            
Amadou    rap                  217
          r&b                  191
          melodic rap          186
          rage rap              78
          hip hop               49
Logan     post-grunge          248
          punk                 146
          alternative metal    142
          hard rock            136
          pop punk             134
Name: count, dtype: int64

In [ ]:
# Keep only each listener's own top 5 genres
top_pairs = set(top_genres.index)

genre_summary = (
    genre_df[
        [
            pair in top_pairs
            for pair in zip(genre_df["Listener"], genre_df["Genre"])
        ]
    ]
    .groupby(["Listener", "Genre"])[["Energy", "Valence"]]
    .mean()
    .reset_index()
)

genre_summary

,Listener,Genre,Energy,Valence
0,Amadou,hip hop,0.711531,0.571010
1,Amadou,melodic rap,0.595468,0.427674
2,Amadou,r&b,0.529257,0.416647
3,Amadou,rage rap,0.631346,0.411831
4,Amadou,rap,0.591857,0.410421
5,Logan,alternative metal,0.878549,0.301368
6,Logan,hard rock,0.890654,0.322054
7,Logan,pop punk,0.847194,0.527493
8,Logan,post-grunge,0.827427,0.348184
9,Logan,punk,0.876952,0.536136


In [30]:
fig = px.bar(
    genre_summary,
    x="Genre",
    y=["Energy", "Valence"],
    barmode="group",
    facet_col="Listener",
    title="Average Energy and Valence by Top Genres"
)

fig.update_xaxes(matches=None)
fig.show()

The graph shows that the top genres have different average energy and valence levels. Hip hop has the highest average energy and valence, while R&B has the lowest average energy. Rap, melodic rap, and rage rap are closer together, but still show some differences. This suggests that audio features like energy and valence may help us compare genres and could be useful later when looking for possible subgenres.